# Test Train TinyLlama HelpSteer2 Adapters

This notebook is a test variant of Notebook 2. It lets you set the number of selected training examples per attribute and the eval-loss check interval before starting an adapter run.


## 1. Clone or update the repository

In [ ]:
%cd /content
import os
import shutil

repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"

if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

## 2. Check the GPU

In [ ]:
!nvidia-smi

## 3. Install dependencies

The pandas and NumPy versions are pinned for compatibility with the standard Colab environment. ArmoRM declares Transformers 4.40.0 and its custom model code relies on that version's internal Llama API, so Transformers, PEFT, and Accelerate are pinned to a compatible set. TinyLlama uses standard LoRA training without TorchAO or 4-bit quantization. Colab may include an old optional `torchao` package, so the install cell removes it.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U "pandas==2.2.2" "numpy<2.1" "transformers==4.40.0" "peft==0.10.0" "accelerate==0.29.3" datasets pyyaml tensorboard

Restart the runtime once after installation so Python forgets previously imported Transformers, TorchAO, or PyTorch modules. This restart is required when replacing a newer Transformers version. Then rerun the repository cell and continue with the GPU and validation cells; you do not need to reinstall the dependencies again in the same Colab session.

## 4. Show important files

In [ ]:
!pwd
!ls
!ls configs
!ls scripts
!ls src

## 5. Test settings

Change these two values before starting training. `TRAIN_EXAMPLES_PER_ATTRIBUTE` controls how many top-rated training texts are used for each adapter. `EVAL_LOSS_CHECK_STEPS` controls how often `eval_loss` is computed and logged.


In [ ]:
TRAIN_EXAMPLES_PER_ATTRIBUTE = 8434
EVAL_LOSS_CHECK_STEPS = 100

if TRAIN_EXAMPLES_PER_ATTRIBUTE < 1:
    raise ValueError("TRAIN_EXAMPLES_PER_ATTRIBUTE must be at least 1.")
if EVAL_LOSS_CHECK_STEPS < 1:
    raise ValueError("EVAL_LOSS_CHECK_STEPS must be at least 1.")

print(f"Training examples per attribute: {TRAIN_EXAMPLES_PER_ATTRIBUTE}")
print(f"Eval loss check every {EVAL_LOSS_CHECK_STEPS} steps")


## 6. Validate the config and inspect HelpSteer2

In [ ]:
!python scripts/validate_tinyllama_helpsteer2_config.py
!python scripts/inspect_helpsteer2_dataset.py --split "train" --max_examples {TRAIN_EXAMPLES_PER_ATTRIBUTE}

## 7. Start TensorBoard

Open the **Scalars** view to inspect training and evaluation loss against `global_step`. New points appear as training writes event logs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/tensorboard/tinyllama_helpsteer2

## 8. Manual one-adapter-at-a-time training

Each code block below starts exactly one adapter as a background process, so the stop and status cells remain usable. Training reads the official HelpSteer2 `train` split and then selects the number of top-rated examples per attribute set in `TRAIN_EXAMPLES_PER_ATTRIBUTE`, while evaluation uses the separate official `validation` split. Start the next adapter manually only after the status cell shows that training has finished and you have checked the previous adapter's `train_loss` and `eval_loss` in TensorBoard. The first line of each training cell removes stale stop files from an earlier run.

LoRA rank is fixed at `r=8` in the script, LoRA dropout is fixed at `0.1`, and `weight_decay=0.01` provides mild AdamW regularization. The first 6% of optimizer steps linearly warm the learning rate from near zero to `1e-4`. After warmup, `cosine_decay` smoothly decays the learning rate to the floor at `0.1` of the initial learning rate (`1e-5`).

Training uses response-only SFT labels: prompt tokens up to and including the `Assistant:` marker are masked with `-100`, so `train_loss` and `eval_loss` are computed only on assistant response tokens. These loss curves are not directly comparable to older full prompt-response loss runs.

ArmoRM monitoring is temporarily disabled in the training commands below. To enable it later, add `--use_armorm_monitoring` and the reward-monitoring arguments again.

### Helpfulness

In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!nohup python -u scripts/train_tinyllama_helpsteer2_adapters.py \
  --attributes helpfulness \
  --split "train" \
  --eval_split "validation" \
  --max_training_examples {TRAIN_EXAMPLES_PER_ATTRIBUTE} \
  --num_epochs 5 \
  --batch_size 8 \
  --max_length 1024 \
  --learning_rate 1e-4 \
  --lr_scheduler_type cosine_decay \
  --min_lr_ratio 0.1 \
  --warmup_ratio 0.06 \
  --weight_decay 0.01 \
  --logging_steps 10 \
  --eval_steps {EVAL_LOSS_CHECK_STEPS} \
  --save_steps 500 \
  --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py --attributes helpfulness

### Correctness

In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!nohup python -u scripts/train_tinyllama_helpsteer2_adapters.py \
  --attributes correctness \
  --split "train" \
  --eval_split "validation" \
  --max_training_examples {TRAIN_EXAMPLES_PER_ATTRIBUTE} \
  --num_epochs 5 \
  --batch_size 8 \
  --max_length 1024 \
  --learning_rate 1e-4 \
  --lr_scheduler_type cosine_decay \
  --min_lr_ratio 0.1 \
  --warmup_ratio 0.06 \
  --weight_decay 0.01 \
  --logging_steps 10 \
  --eval_steps {EVAL_LOSS_CHECK_STEPS} \
  --save_steps 500 \
  --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py --attributes correctness

### Coherence

In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!nohup python -u scripts/train_tinyllama_helpsteer2_adapters.py \
  --attributes coherence \
  --split "train" \
  --eval_split "validation" \
  --max_training_examples {TRAIN_EXAMPLES_PER_ATTRIBUTE} \
  --num_epochs 5 \
  --batch_size 8 \
  --max_length 1024 \
  --learning_rate 1e-4 \
  --lr_scheduler_type cosine_decay \
  --min_lr_ratio 0.1 \
  --warmup_ratio 0.06 \
  --weight_decay 0.01 \
  --logging_steps 10 \
  --eval_steps {EVAL_LOSS_CHECK_STEPS} \
  --save_steps 500 \
  --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py --attributes coherence

### Complexity

In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!nohup python -u scripts/train_tinyllama_helpsteer2_adapters.py \
  --attributes complexity \
  --split "train" \
  --eval_split "validation" \
  --max_training_examples {TRAIN_EXAMPLES_PER_ATTRIBUTE} \
  --num_epochs 5 \
  --batch_size 8 \
  --max_length 1024 \
  --learning_rate 1e-4 \
  --lr_scheduler_type cosine_decay \
  --min_lr_ratio 0.1 \
  --warmup_ratio 0.06 \
  --weight_decay 0.01 \
  --logging_steps 10 \
  --eval_steps {EVAL_LOSS_CHECK_STEPS} \
  --save_steps 500 \
  --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py --attributes complexity

### Verbosity

In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!nohup python -u scripts/train_tinyllama_helpsteer2_adapters.py \
  --attributes verbosity \
  --split "train" \
  --eval_split "validation" \
  --max_training_examples {TRAIN_EXAMPLES_PER_ATTRIBUTE} \
  --num_epochs 5 \
  --batch_size 8 \
  --max_length 1024 \
  --learning_rate 1e-4 \
  --lr_scheduler_type cosine_decay \
  --min_lr_ratio 0.1 \
  --warmup_ratio 0.06 \
  --weight_decay 0.01 \
  --logging_steps 10 \
  --eval_steps {EVAL_LOSS_CHECK_STEPS} \
  --save_steps 500 \
  --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py --attributes verbosity

## 9. Stop the current adapter

Run this cell while background training is active if you want to stop the current adapter. The script does not wait until the end of the epoch. It stops after the next save step, saves the adapter, removes `STOP_CURRENT_ADAPTER`, and exits gracefully. With `--save_steps 500`, the maximum wait is roughly until the next 500-step checkpoint.

In [ ]:
!touch STOP_CURRENT_ADAPTER

## 10. Check the running training

Run these cells at any time to see whether an adapter process is active and to inspect the latest training output. The adapter commands write to `/content/tinyllama_helpsteer2_training.log`.

In [ ]:
!pgrep -af "[t]rain_tinyllama_helpsteer2_adapters.py" || echo "No adapter training process is running."
!latest_epoch=$(grep -E "Epoch [0-9]+/[0-9]+.*shuffle_seed=" /content/tinyllama_helpsteer2_training.log 2>/dev/null | tail -n 1); [ -n "$latest_epoch" ] && echo "$latest_epoch" || echo "No epoch seed found yet."
!tail -n 30 /content/tinyllama_helpsteer2_training.log 2>/dev/null || echo "No training log exists yet."

In [ ]:
!pgrep -af "[t]rain_tinyllama_helpsteer2_adapters.py" || echo "No adapter training process is running."
!grep -i "eval_loss" /content/tinyllama_helpsteer2_training.log 2>/dev/null | tail -n 30 || echo "No eval_loss found yet."

In [ ]:
!pgrep -af "[t]rain_tinyllama_helpsteer2_adapters.py" || echo "No adapter training process is running."
!grep "objective=helpsteer-helpfulness" /content/tinyllama_helpsteer2_training.log 2>/dev/null | tail -n 30 || echo "No helpsteer-helpfulness ArmoRM monitoring found yet."

Check the ArmoRM download cache if training is currently loading the reward monitor.

In [ ]:
# Check ArmoRM Download
!du -sh /root/.cache/huggingface/hub/models--RLHFlow--ArmoRM-Llama3-8B-v0.1 2>/dev/null || echo "ArmoRM cache not created yet."
!ls -lh /root/.cache/huggingface/hub/models--RLHFlow--ArmoRM-Llama3-8B-v0.1/snapshots/*/ 2>/dev/null || echo "ArmoRM snapshot not created yet."

## 11. Inspect saved logs

Use these folders to inspect the CSV logs and TensorBoard event files.

In [ ]:
!ls results/tinyllama_helpsteer2_training_logs
!ls results/tensorboard/tinyllama_helpsteer2

## 12. Export training curves

In [ ]:
!python scripts/export_tinyllama_training_curves.py || true

## 13. Final adapter check

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py

## 14. Create a local adapter backup

Run this only after all five adapters are finished. The zip file is a generated local backup and must stay out of Git.

In [ ]:
!zip -r tinyllama_helpsteer2_adapters.zip adapters/tinyllama-helpsteer2-*-adapter/

## 15. Git safety check

Adapters, checkpoints, safetensors, `.bin` files, model weights, TensorBoard events, and zip files are generated artifacts. Keep them out of Git unless a small result file is intentionally selected later.

In [ ]:
!git status